In [3]:
!pip install torch torchvision torchaudio


  Using cached torch-2.9.0-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.24.0-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/109.3 MB ? eta -:--:--
   ---------------------------------------- 1.0/109.3 MB 11.2 MB/s eta 0:00:10
   ---------------------------------------- 1.3/109.3 MB 4.8 MB/s eta 0:00:23
    --------------------------------------- 1.8/109.3 MB 2.5 MB/s eta 0:00:43
    --------------------------------------- 2.1/109.3 MB 2.2 MB/s eta 0:00:50
    --------------------------------------- 2.1/109.3 MB 2.2 MB/s eta 0:00:50
    --------------------------------------- 2.4/109.3 MB 2.0 MB/s eta 0:00:55
    --------------------------------------- 2.6/109.3 MB 1.7 MB/s eta 0:01:02
   - -------------------------------------- 2.9/109.3 MB 1.7 MB/s eta 0:01:04
   - -------------------------------------- 3.1/109.3 MB 1.6 MB/s eta 0:01:07
   - --

In [8]:
!pip install numpy


In [9]:
!pip install facenet-pytorch


  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + E:\anaconda3\python.exe C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf7903e95dbf8\vendored-meson\meson\meson.py setup C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf7903e95dbf8 C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf7903e95dbf8\.mesonpy-imf1_yzx -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf7903e95dbf8\.mesonpy-imf1_yzx\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf7903e95dbf8
      Build dir: C:\Users\hp lap\AppData\Local\Temp\pip-install-2zw77jbe\numpy_b70a5e4efb4c4994a82bf790

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
##from facenet_pytorch import MTCNN
from sklearn.decomposition import PCA
from PIL import Image
import numpy as np

In [11]:
# 1. Preprocessing: Face Detection and Alignment
def preprocess_face(image_path, size=299):
    mtcnn = MTCNN(image_size=size, margin=0)
    img = Image.open(image_path).convert('RGB')
    face = mtcnn(img)
    if face is not None:
        return face
    transform = transforms.Compose([transforms.Resize(size), transforms.CenterCrop(size), transforms.ToTensor()])
    return transform(img)

In [12]:
# 2. Detail Stream: PCA-Based Texture Feature Extraction
def extract_texture_features(face_img, n_components=50):
    np_face = face_img.permute(1, 2, 0).cpu().numpy()  # (H, W, C)
    flat_pixels = np_face.reshape(-1, np_face.shape[-1])
    pca = PCA(n_components=n_components)
    texture_features = pca.fit_transform(flat_pixels)
    output = texture_features.reshape(np_face.shape[0], np_face.shape[1], -1)
    tensor_output = torch.tensor(output).permute(2, 0, 1).float()  # (components, H, W)
    return tensor_output

In [13]:
# 3. Feature Extractor Modules
class RGBFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()
    def forward(self, x):
        return self.backbone(x)

class PCAFeatureExtractor(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((16, 16))
    def forward(self, x):
        x = self.reduce(x)
        x = self.pool(x)
        return x

In [14]:
# 4. Granular Spatial Attention
class GranularSpatialAttention(nn.Module):
    def __init__(self, in_ch, mask_size=16):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 1, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(mask_size)
    def forward(self, feat):
        avg = self.avg_pool(feat)
        mask = torch.sigmoid(self.conv(avg))  # (B, 1, mask_size, mask_size)
        mask_up = F.interpolate(mask, size=feat.shape[2:], mode='bilinear', align_corners=False)
        attended = feat * mask_up
        return attended, mask_up

In [15]:
# 5. Fusion and Classifier
class FeatureFusionClassifier(nn.Module):
    def __init__(self, rgb_ch=512, detail_ch=128, spatial_dim=16):
        super().__init__()
        self.rgb_attn = GranularSpatialAttention(rgb_ch, mask_size=spatial_dim)
        self.detail_attn = GranularSpatialAttention(detail_ch, mask_size=spatial_dim)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(rgb_ch + detail_ch, 128),
            nn.ReLU(),
            nn.Linear(128, 2),    # binary softmax
        )
    def forward(self, rgb_feat, detail_feat):
        rgb_att, rgb_mask = self.rgb_attn(rgb_feat)
        detail_att, detail_mask = self.detail_attn(detail_feat)
        rgb_flat = self.global_pool(rgb_att).flatten(1)
        detail_flat = self.global_pool(detail_att).flatten(1)
        fusion = torch.cat([rgb_flat, detail_flat], dim=1)
        logits = self.fc(fusion)
        return logits, rgb_mask, detail_mask

In [16]:
# 6. Full Model Integration
class ForgeryDetectionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_extractor = RGBFeatureExtractor()
        self.detail_extractor = PCAFeatureExtractor(50, 128)  # 50 PCA components, out to 128
        self.fuser = FeatureFusionClassifier(rgb_ch=512, detail_ch=128)
    def forward(self, rgb_input, detail_input):
        rgb_feat = self.rgb_extractor(rgb_input)              # (B, 512)
        detail_feat = self.detail_extractor(detail_input)     # (B, 128, 16, 16)
        logits, rgb_mask, detail_mask = self.fuser(rgb_feat.unsqueeze(-1).unsqueeze(-1), detail_feat)
        return logits, rgb_mask, detail_mask

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from sklearn.decomposition import PCA
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader

# 1. Preprocessing: Resize & Tensor Conversion
def preprocess_face(image_path, size=299):
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor()
    ])
    return transform(img)  # shape: (3, size, size)

# 2. PCA-Based Texture Feature Extraction with capped components
def extract_texture_features(face_img, n_components=3):
    np_face = face_img.permute(1, 2, 0).cpu().numpy()  # (H, W, C)
    flat_pixels = np_face.reshape(-1, np_face.shape[-1])
    n_components = min(n_components, flat_pixels.shape[1], flat_pixels.shape[0])
    pca = PCA(n_components=n_components)
    texture_features = pca.fit_transform(flat_pixels)
    output = texture_features.reshape(np_face.shape[0], np_face.shape[1], -1)
    tensor_output = torch.tensor(output).permute(2, 0, 1).float()  # (components, H, W)
    return tensor_output

# 3. Feature Extractors
class RGBFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()
    def forward(self, x):
        return self.backbone(x)

class PCAFeatureExtractor(nn.Module):
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((16, 16))
    def forward(self, x):
        x = self.reduce(x)
        x = self.pool(x)
        return x

# 4. Granular Spatial Attention
class GranularSpatialAttention(nn.Module):
    def __init__(self, in_ch, mask_size=16):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 1, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(mask_size)
    def forward(self, feat):
        avg = self.avg_pool(feat)
        mask = torch.sigmoid(self.conv(avg))
        mask_up = F.interpolate(mask, size=feat.shape[2:], mode='bilinear', align_corners=False)
        attended = feat * mask_up
        return attended, mask_up

# 5. Fusion & Classification Module
class FeatureFusionClassifier(nn.Module):
    def __init__(self, rgb_ch=512, detail_ch=128, spatial_dim=16):
        super().__init__()
        self.rgb_attn = GranularSpatialAttention(rgb_ch, mask_size=spatial_dim)
        self.detail_attn = GranularSpatialAttention(detail_ch, mask_size=spatial_dim)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(rgb_ch + detail_ch, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )
    def forward(self, rgb_feat, detail_feat):
        rgb_att, _ = self.rgb_attn(rgb_feat)
        detail_att, _ = self.detail_attn(detail_feat)
        rgb_flat = self.global_pool(rgb_att).flatten(1)
        detail_flat = self.global_pool(detail_att).flatten(1)
        fusion = torch.cat([rgb_flat, detail_flat], dim=1)
        logits = self.fc(fusion)
        return logits

# 6. Complete Forgery Detection Model
class ForgeryDetectionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_extractor = RGBFeatureExtractor()
        self.detail_extractor = PCAFeatureExtractor(3, 128)  # 3 PCA components now max
        self.fuser = FeatureFusionClassifier(rgb_ch=512, detail_ch=128)
    def forward(self, rgb_input, detail_input):
        rgb_feat = self.rgb_extractor(rgb_input)
        detail_feat = self.detail_extractor(detail_input)
        logits = self.fuser(rgb_feat.unsqueeze(-1).unsqueeze(-1), detail_feat)
        return logits

# Dataset Class
class ForgeryDataset(Dataset):
    def __init__(self, real_dir, fake_dir, pca_components=3, im_size=299):
        self.samples = []
        self.labels = []
        self.pca_components = pca_components
        self.size = im_size
        for imgname in os.listdir(real_dir):
            if imgname.lower().endswith(('.jpg', '.png')):
                self.samples.append(os.path.join(real_dir, imgname))
                self.labels.append(1)
        for imgname in os.listdir(fake_dir):
            if imgname.lower().endswith(('.jpg', '.png')):
                self.samples.append(os.path.join(fake_dir, imgname))
                self.labels.append(0)
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        rgb_img = preprocess_face(img_path, size=self.size)
        detail = extract_texture_features(rgb_img, n_components=self.pca_components)
        return rgb_img, detail, torch.tensor(label, dtype=torch.long)

# Replace these with your dataset paths
real_dir = 'E:/ROP_Project/CASIA2/Au'
fake_dir = 'E:/ROP_Project/CASIA2/Tp'

dataset = ForgeryDataset(real_dir, fake_dir)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ForgeryDetectionModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for rgb_input, detail_input, label in train_loader:
        rgb_input = rgb_input.to(device)
        detail_input = detail_input.to(device)
        label = label.to(device)
        logits = model(rgb_input, detail_input)
        loss = criterion(logits, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

print("Training complete!")


e:\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/5 - Loss: 0.4578
Epoch 2/5 - Loss: 0.3995
Epoch 3/5 - Loss: 0.3674


KeyboardInterrupt: 